# Predicción climática Huánuco — entrenamiento en Google Colab

Este notebook corre las etapas pesadas del pipeline (**05_modelar.py** y **06_mapas_zonas.py**) en Colab.
El saneamiento y la unión con PISCO/ONI ya se hicieron en la Mac (`dataset_modelo.csv`).

**Pasos:**
1. Ejecuta la celda 1 y sube `paquete_colab.zip` (está en la carpeta `colab/` del proyecto).
2. Ejecuta las celdas en orden (*Entorno de ejecución → Ejecutar todas* también sirve).
3. Al final se descarga `resultados.zip`: descomprímelo en la carpeta del proyecto en la Mac y corre `streamlit run app.py`.

No hace falta GPU: el entorno CPU estándar alcanza (unos 50 000 filas).

## 1. Subir el paquete

In [ ]:
from google.colab import files
import zipfile, os
subido = files.upload()                      # elige paquete_colab.zip
nombre = next(iter(subido))
zipfile.ZipFile(nombre).extractall('.')
!ls -la

## 2. Dependencias
Colab ya trae pandas, scikit-learn, xgboost, matplotlib y joblib; esto solo confirma versiones.

In [ ]:
!pip -q install xgboost scikit-learn joblib matplotlib
import pandas, sklearn, xgboost, os
print('pandas', pandas.__version__, '| sklearn', sklearn.__version__, '| xgboost', xgboost.__version__, '| CPUs', os.cpu_count())

## 3. Modelado (objetivo principal: PISCOp)
Climatología, Persistencia, Random Forest, XGBoost y MLP para h = 1, 2, 3; ablación; importancia por permutación; pronóstico actual.

In [ ]:
!python 05_modelar.py --datos dataset_modelo.csv --objetivo pp_pisco

## 4. Mapas y zonas climáticas

In [ ]:
!python 06_mapas_zonas.py

## 5. Revisar resultados

In [ ]:
import pandas as pd
from IPython.display import Image, display
met = pd.read_csv('resultados/metricas.csv')
display(met[met.particion == 'test'].pivot_table(index='modelo', columns='h', values=['RMSE', 'SS_clim', 'KGE']).round(3))
display(pd.read_csv('resultados/ablacion.csv').round(3))
display(pd.read_csv('resultados/importancia.csv').head(10))
display(pd.read_csv('resultados/resumen_zonas.csv'))
for m in ['mapa_climatologia', 'mapa_variabilidad', 'mapa_error_modelo', 'mapa_pronostico', 'mapa_zonas']:
    display(Image(f'resultados/{m}.png'))

## 6. Descargar resultados a la Mac

In [ ]:
!zip -qr resultados.zip resultados
files.download('resultados.zip')

## 7. (Opcional) Variantes
Cada variante sobrescribe `resultados/`, así que primero se renombra la carpeta. Ejecuta solo si las necesitas.

In [ ]:
# Variante A: CHIRPS como objetivo (comparación con la corrida de prueba de la documentación)
# !mv resultados resultados_pisco
# !python 05_modelar.py --datos dataset_huanuco_2015_2025.csv --objetivo pp_chirps && python 06_mapas_zonas.py
# !mv resultados resultados_chirps && mv resultados_pisco resultados

# Variante B: temperatura media (t2m)
# !mv resultados resultados_pisco
# !python 05_modelar.py --datos dataset_modelo.csv --objetivo t2m && python 06_mapas_zonas.py
# !mv resultados resultados_t2m && mv resultados_pisco resultados

# !zip -qr variantes.zip resultados_chirps resultados_t2m 2>/dev/null; files.download('variantes.zip')